In [ ]:
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [ ]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)
x_totem, y_totem, yerr_totem = process_data(totem_data, totem_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]


In [ ]:
n = 3000

b_0 = (33 - 6) / (12 * np.pi)
lambda_qcd = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



param_mg_atlas_pl = 0.421
param_eps_atlas_pl = 0.0753
param_a1_atlas_pl = 1.517
param_a2_atlas_pl = 2.05

# # Get parameters for selected configuration
# initial_params_log_atlas = ensemble_parameters['atlas']['log']
# initial_params_pl_atlas = ensemble_parameters['atlas']['pl']


In [ ]:
#--------------------------------------
# Eq 22 - GE
#--------------------------------------
def m2_pl(q2, mg):
    lambda2 = lambda_qcd ** 2
    rho_mg_2 = rho * (mg ** 2)
    ratio = np.log((q2 + rho_mg_2) / lambda2) / np.log(rho_mg_2 / lambda2)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

#--------------------------------------
# Eq 26 - GE
#--------------------------------------
def G_p(q2, a1, a2):
    t = -q2
    return np.exp(-(a1 * np.abs(t) + a2 * np.abs(t) ** 2))


#--------------------------------------
# Eq 24 - GE
#--------------------------------------
def alpha_D(q2, mg, m2_type):
    m2_func = m2_type(q2, mg)
    return 1.0 / (b_0 * (q2 + m2_func) * np.log((q2 + 4 * m2_func) / (lambda_qcd ** 2)))

#--------------------------------------
# Eq 7 - GE
#--------------------------------------
def T_1(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    G0 = G_p(q, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2


#--------------------------------------
# Eq 8 - GE
#--------------------------------------
def T_2(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


#--------------------------------------
# Eq 11 - GE
#--------------------------------------
def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

#--------------------------------------
# Eq 6 - GE
#--------------------------------------
def born_amp(diff_T, s, eps, t):
    alpha_pomeron = 1.0 + eps + 0.25 * t
    s_tilde = s/s0

    return 1j * s * 8 * (s_tilde**(alpha_pomeron - 1)) * diff_T

In [ ]:
def k_integral(phi, mg, a1, a2, m2_func, q, k_max, n=n):
    """
    PURPOSE: Compute the inner integral over k for a fixed φ (phi) value.
    Returns the complex value of the k-integral.
    """
    def integrand(k):
        """Integrand for k integration"""
        result = k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                      T_2(k, phi, mg, a1, a2, m2_func, q))
        return result
    
    # Use fixed_quad with direct limits [0, k_max]
    result, _ = fixed_quad(integrand, 0, k_max, n=n)
    return result

def phi_integral(mg, a1, a2, m2_func, q, k_max, n=20):
    """
    PURPOSE: Compute the outer integral over φ (phi) from 0 to 2π.
    Calls k_integral for each phi value.
    Returns the complex value of the phi-integral.
    """
    def integrand(phi):
        """Integrand for phi integration"""
        return k_integral(phi, mg, a1, a2, m2_func, q, k_max, n)
    
    # Use fixed_quad with direct limits [0, 2π]
    result, _ = fixed_quad(integrand, 0, 2*np.pi, n=n)
    return result

def compute_k_phi_integral(mg, a1, a2, m2_func, q, k_max, n=n):

    return phi_integral(mg, a1, a2, m2_func, q, k_max, n)

# TESTING 
mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0
k_max = 13000

# Test the integration
result = compute_k_phi_integral(mg, a1, a2, m2_pl, q, k_max)
print(f"Complex result: {result}")

In [ ]:
from iminuit import Minuit

def f(a, b):
    return (a-2)**2 + (b+1)**2

m = Minuit(f, a=0, b=0)
m.migrad()

print("EDM =", m.fmin.edm)
print("Convergiu? ", m.fmin.is_valid)


In [ ]:
# start_sqrt_s = 100
# end_sqrt_s = 13001
# step_size = 100
# def born_sigma_tot(amp_value, s):
#     return amp_value.imag / s * 0.389379323
# lst = []
# s = []
# current_sqrt_s = start_sqrt_s
# while current_sqrt_s <= end_sqrt_s:
#     current_s = current_sqrt_s**2 
#     diff_t = compute_k_phi_integral(
#         k_max=current_sqrt_s,
#         mg= param_mg_atlas_pl,
#         a1= param_a1_atlas_pl,
#         a2= param_a2_atlas_pl,
#         m2_func= m2_pl, 
#         q = 0
#     )
#     born_amp_value = born_amp(diff_t,
#                               current_s,
#                               param_eps_atlas_pl,
#                               0)
#     current_sqrt_s += step_size
#     # print(born_amp_value) 
#     lst.append(born_sigma_tot(born_amp_value, current_s))
#     s.append(current_sqrt_s)
    
#     # print(born_sigma_tot(born_amp_value, current_s))

# data_sigma_tot_atlas = pd.read_csv(
#     "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
#     delim_whitespace=True,
#     header=None,
#     nrows=70
# )
# x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
# y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
# y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()


# def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
#     fig.add_trace(go.Scatter(
#         x=x,
#         y=y,
#         mode='lines+markers',
#         line=dict(color=color, width=width, dash=line_style),
#         marker=dict(size=size),
#         name = label, 
#         showlegend=legend
#     ))
# def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
#                            name=None, show_label=True, mode='markers'):
#     fig.add_trace(go.Scatter(
#         x=x,
#         y=y * scale,
#         mode=mode,
#         marker=dict(color=color, size=size),
#         error_y=dict(
#             type='data',
#             array=y_error * scale,
#             visible=True
#         ),
#         name=name if show_label else None,
#         showlegend=show_label
#     ))

# fig = go.Figure()


# add_total_trace(fig, s, lst, color='blue', label='PL Atlas', line_style='solid')
# add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

# fig.update_layout(
#     title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
#     xaxis=dict(
#         title='√s [GeV]',
#         type='log',
#         range=[np.log10(2000), np.log10(14000)],
#     ),
#     yaxis=dict(
#         title='σ_tot [mb]',
#         range=[80, 125]
#     ),
#     showlegend=True,
#     legend=dict(
#         title='Ensembles'
#     ),
#     plot_bgcolor='white',
#     hovermode='x unified'
# )
    
# fig.update_xaxes(gridcolor='lightgray')
# fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")

In [ ]:
#--------------------------------------
# Eq 23 - EIK
#--------------------------------------
def chi_integral(sqrt_s_val, b, q, mg, a1, a2, m2_func, born_amp_value, eps, n = n):

    # q = np.sqrt(-t)

    s_val = sqrt_s_val**2

    # diff_t = compute_k_phi_integral(
    #     mg, a1, a2, m2_func, q, sqrt_s_val, n)

    # t = -(q**2)
    # born_amp_value = born_amp(diff_t, s_val, eps, t)

    def q_integrand(q):
        return (q * j0(b * q) * born_amp_value) / s_val
    
    result, _ = fixed_quad(q_integrand, 0, 0.2, n=n)

    return result

# Testing chi_integral
# print(chi_integral(13000, 30, 0.2, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl, m2_pl, param_eps_atlas_pl))

In [ ]:
#--------------------------------------
# Eq 24 - EIK
#--------------------------------------
def eik_amp(s, q, chi):
    factor = 1 - np.exp(1j * chi)

    def b_integrand(b):
        return b * j0(b * q) * factor * (1j * s)
    
    result, _ = fixed_quad(b_integrand, 0, 30, n=n)

    return result

# testing eik_amp
# print(eik_amp(13000, 30, 0.2, -0.48j))


In [ ]:
#--------------------------------------
# Eq 12 - GE
#--------------------------------------
def born_dif_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323

#--------------------------------------
# Eq 14 - EIK
#--------------------------------------
def eik_dif_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    factor = np.pi/(s**2)
    return factor * amp_squared

In [ ]:
# lista de valores de q para a integral eikonal
lst_q_integration = np.linspace(0, 0.2, 100)
lst_b_integration = np.linspace(0, 10, 100)

# escolha do modelo de massa
mass_model = 'pl'
# m2_func = m2_pl if mass_model == 'pl' else m2_log


In [ ]:
def model_function(x_born, eps, mg, a1, a2, sqrt_s, model_type='log'):

    s = sqrt_s ** 2
    results = []    

    # loop sobre os pontos experimentais (para cada t)
    for q_exp in x_born:

        eik_amp_sum = 0 

        for b_value in lst_b_integration:
            chi_sum = 0
            
            for q_int in lst_q_integration:

                diff_t = compute_k_phi_integral(mg, a1, a2, m2_pl, q_int, sqrt_s)
                
                t = -(q_int**2)
                born_amp_value = born_amp(diff_t, s, eps, t)
                chi_value = (q_int * j0(b_value * q_int) * born_amp_value) / s

                # chi_value = chi_integral(sqrt_s, b_value, q_int, mg, a1, a2, m2_pl, eps)
                
                chi_sum += chi_value   

            factor = 1 - np.exp(1j * chi_sum)
            eik_amp_value = b_value * j0(b_value * q_exp) * factor * (1j * s)
            eik_amp_sum += eik_amp_value
            
        # eik_amp_value = eik_amp(s, 30, q_exp, chi_value)

        # seção de choque diferencial
        diff_sigma = born_dif_sigma(eik_amp_sum, s)
        results.append(diff_sigma)

    return np.array(results)


In [ ]:
def make_least_squares(x, y, yerr, energy, model_type):
    return LeastSquares(x, y, yerr, 
        lambda x, eps, mg, a1, a2: model_function(x, eps, mg, a1, a2, energy, model_type))

total_cost_pl_atlas = (
    make_least_squares(x_7_atlas, y_7_atlas, yerr_7_atlas, 7000, 'pl')
)


In [ ]:
def otimization(total_cost_func, mg_init, eps_init, a1_init, a2_init, model_type: str, ensemble: str):
    print('\n')
    print(80 * '-')
    print(f"Iniciando otimização dos parâmetros usando LeastSquares para {model_type} em {ensemble.upper()}")

    m = Minuit(total_cost_func, 
               eps=eps_init,
               mg=mg_init,
               a1=a1_init,
               a2=a2_init)

    m.strategy = 2
    m.errordef = 1
    m.tol = 0.01
    # m.precision = 1e-3
    
    down = 0.4
    up = 1.5

    m.limits['mg'] = (down * param_mg_atlas_pl, up * param_mg_atlas_pl)
    m.limits['eps'] = (down * param_eps_atlas_pl, up * param_eps_atlas_pl)
    m.limits['a1'] = (down * param_a1_atlas_pl, up * param_a1_atlas_pl)
    m.limits['a2'] = (down * param_a2_atlas_pl, up * param_a2_atlas_pl)



    m.simplex()
    m.simplex()

    m.migrad()

    

    count = 0
    if m.valid == False:
        while m.valid == False:
            print(f'Rodando migrad novamente tentativa {count+1}')
            m.migrad()
            count += 1
            if count >= 10:
                print(f'Minuit inválido após {count} tentativas.')
                break
    else:
        print("Minuit validado com sucesso!")



    edm_goal = 1e-7
    count = 0
    if m.fmin.edm > edm_goal:
        while m.fmin.edm > edm_goal:
            print(f'EDM maior que o objetivo ({edm_goal}), rodando mais migrad tentativa {count + 1}')
            m.simplex()
            m.migrad()
            count += 1
            if count >= 10:
                print(f'O EDM não caiu abaixo do objetivo após {count} tentativas.')
                break
    else:
        print('O EDM está dentro do objetivo')


    m.hesse()
    m.minos(cl = 0.9)

    print(f"Parâmetros otimizados para {model_type} em {ensemble}: \n")
    print(f'mg: {m.values["mg"]} ± {m.errors["mg"]}')
    print(f'eps: {m.values["eps"]} ± {m.errors["eps"]}')
    print(f'a1: {m.values["a1"]} ± {m.errors["a1"]}')
    print(f'a2: {m.values["a2"]} ± {m.errors["a2"]}')
    print(f'chi2/ndof: {m.fval / m.ndof}')


    return m



In [ ]:
m_pl_atlas = otimization(total_cost_pl_atlas,
                         mg_init = param_mg_atlas_pl, 
                         eps_init = param_eps_atlas_pl, 
                         a1_init = param_a1_atlas_pl, 
                         a2_init = param_a2_atlas_pl, 
                         model_type = 'pl', 
                         ensemble = 'atlas')


# mg: 0.23411637728741208 ± 0.00033574643816519965
# eps: 0.12685040781076112 ± 0.001340347140610439
# a1: 1.1692659320733771 ± 0.01474587351542503
# a2: 0.6767754519998356 ± 0.03910298646587622
# chi2/ndof: 350.38756456977086